## Loading black-box functions in MPS using Tensor Cross-Interpolation

In this notebook, we show how to use tensor cross-interpolation (TCI), also known as TT-Cross, to load generic black-box functions in MPS. TT-Cross is an algorithm that samples a black-box function along a collection of elements known as *crosses* to find the collection of MPS tensors that minimizes the error. There are several variants of TT-Cross in the literature. Currently, this library implements the following variants:
1. *maxvol*-based cross interpolation (`cross_maxvol`).
2. *DMRG*-based cross interpolation (`cross_DMRG`).
3. *greedy* cross interpolation (`cross_greedy`) (in development).

These black-box functions take many forms, and currently the following applications have been implemented:
- Load multivariate functions in MPS (using `BlackBoxLoadMPS`).
- Load multivariate functions in tensor trains (similar to MPS but without quantization, using `BlackBoxLoadTT`).
- Load bivariate functions in MPO (using `BlackBoxLoadMPO`).
- Compose multivariate functions with collections of MPS (using `BlackBoxComposeMPS`).

Any other potential application can be implemented by, for instance, extending the base class `BlackBox`.

In [2]:
import numpy as np
import matplotlib.pyplot as plt

from seemps.analysis.mesh import Mesh, RegularInterval
from seemps.analysis.cross import cross_dmrg, cross_maxvol, CrossStrategyMaxvol
import seemps.tools

seemps.tools.DEBUG = 2

#### 1. Load multivariate functions in MPS

Load a bivariate product Gaussian function as MPS:

$$
f(x, y) = e^{-(x^2 + y^2)}
$$

In [7]:
from seemps.analysis.cross import BlackBoxLoadMPS

func = lambda tensor: np.exp(-np.sum(tensor**2, axis=0))

start, stop = -1, 1
num_qubits = 10
interval = RegularInterval(start, stop, 2**num_qubits)
dimension = 2
mesh = Mesh([interval] * dimension)

black_box = BlackBoxLoadMPS(func, mesh)
mps = cross_dmrg(black_box).mps

 Cross sweep   1 with error(1000 samples in norm-inf)=0.09507421820145934, maxbond=2, evals(cumulative)=144
 Cross sweep   1 with error(1000 samples in norm-inf)=1.3799081311494099e-05, maxbond=4, evals(cumulative)=624
 Cross sweep   2 with error(1000 samples in norm-inf)=5.395683899678261e-14, maxbond=8, evals(cumulative)=1920
 State converged within tolerance 1e-12


#### 2. Load multivariate functions in TT

In [8]:
from seemps.analysis.cross import BlackBoxLoadTT

func = lambda tensor: np.exp(-np.sum(tensor**2, axis=0))

start, stop = -1, 1
num_nodes = 1000
interval = RegularInterval(start, stop, num_nodes)
dimension = 2
mesh = Mesh([interval] * dimension)

black_box = BlackBoxLoadTT(func, mesh)
mps = cross_maxvol(black_box).mps

 Cross sweep   1 with error(1000 samples in norm-inf)=3.3306690738754696e-16, maxbond=3, evals(cumulative)=8000
 State converged within tolerance 1e-12


#### 3. Load bivariate functions in MPO

In [5]:
from seemps.analysis.cross import BlackBoxLoadMPO

#### 4. Compose multivariate functions with collections of MPS

In [6]:
from seemps.analysis.cross import BlackBoxComposeMPS